# Description Deep Dive: Honest Summaries and Visualization

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb06_description_student.ipynb)

---

## 🧭 Approach & Claim Boundary

**Approach emphasis:** description — the first of the four approaches, and the
one every project touches. Before you can infer, predict, or reason about
cause, you have to say plainly what is *in* the data. Doing that honestly is
harder than it looks: a summary is a compression, and every compression can
distort.

| | |
|---|---|
| **A claim this topic PERMITS** | "In these 10,000 resampled LAPOP Brazil interviews, X% report Y, and the distribution looks like Z." |
| **A claim this topic does NOT permit** | "Brazilians believe Y" (generalization beyond the data) or "Y because Z" (causation) — description stops at the data's edge. |

**Where this sits in the course:** meetings 13–14 of 44. It is the description
deep dive that turns the M5 approach-classification skill into real practice,
and it bridges M04 (Measurement) into M05 (Data Strategy) — once you can
summarize data honestly, the next question is *whose* data you are summarizing.

*Provenance: RDSS ch. 15 'Observational: descriptive' + rdss lapop_brazil | observational descriptive designs | dataset shipped as CSV (MIT, attributed); 10k-resample caveat taught | adapted*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Name a variable's **type** (nominal, ordinal, numeric) and choose the
   summary that fits it — and refuse the ones that do not.
2. Read a **distribution**, not just an average, and say what the shape reveals
   that the mean conceals.
3. Report **center and spread** together, and run one **honest group comparison**
   that stops exactly at the data's edge.
4. Recognize and repair the classic **chart distortions** (truncated axis,
   cherry-picked framing) so a figure says no more than its numbers do.
5. Teach yourself the **lapop_brazil resampling caveat** as a provenance lesson —
   what a resampled file is good for and what it would be dishonest to do with.
6. Draft the **Table 1 skeleton** for your own project's future data.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters — and the Provenance Caveat You Owe Every Reader

> *"Show me your Table 1 before you show me your finding. If you cannot
> describe your data honestly — what is in it, who is in it, what shape it
> takes — I will not believe anything you built on top of it."*
> — your **thesis advisor**, on the first draft they will ever read

Description is where trust is won or lost. Every fancy method downstream
inherits whatever distortion you let into the plain summary. So the discipline
starts here, with two habits: say what is in the data, and say where the data
came from.

We will practice on a real survey — 10,000 interviews from the AmericasBarometer
(LAPOP) study of Brazil, bundled with the course textbook. But there is a catch
you must carry openly, and it is itself the first lesson:

> **A question that often comes up here:** *"If this is real survey data, why
> can't I just say what Brazilians think?"* Because this particular file is a
> **10,000-row resample *with replacement*** of the original survey — the
> package built it by drawing rows from the real interviews over and over. That
> makes it perfect for *learning* and *planning* (the patterns are realistic,
> the code runs fast), and dishonest for *substantive claims* (the resampling
> manufactures a precision the original survey never had). Treat every number
> below as a teaching illustration. When you report it, the resampling caveat
> travels with the number — that is what "documenting provenance" means in
> practice.

---

## 2. Variable Types — Because the Wrong Summary Is a Lie in Disguise

**Guiding question:** *before you summarize a column, how do you know which
summaries the column will even let you compute honestly?*

Every dataset shares the same anatomy — the rows are the elements (the units you
observed), the columns are the variables you recorded, and each cell is a single
observation:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/variables_observations.png" width="75%"/></center>

*The anatomy of a data table: rows are the elements (units) you observed, columns
are the variables you recorded, and each cell is one observation. (From Professor
Moreira's QM 67000 Business Analytics slides.)*

A summary only tells the truth if it fits the variable. Averaging a category
code, or reporting a "percentage" of a temperature, produces a number that
looks fine and means nothing. Three types cover almost everything you will meet:

- **Nominal** — labels with no order (a municipality, a party). Summarize with
  counts and the most common category; never with a mean.
- **Ordinal** — ordered categories with unequal or unknown gaps (a 1–7 trust
  scale, a Likert agreement item). Counts and the median are safe; the mean is
  common but leans on the fiction that the gaps are equal.
- **Numeric** — genuine quantities with meaningful gaps (age, income, a count).
  Mean, median, and spread all apply.

> **A question that often comes up here:** *"Everyone means their 1–7 scales —
> can't I just treat an ordinal item as numeric?"* You can, and much of social
> science does, but naming what the shortcut assumes keeps you honest: taking a
> mean treats the step from "1" to "2" as identical to the step from "6" to "7,"
> which an ordinal scale never promised. The safe habit is to report the median
> (and the shape) whenever the mean would lean hard on that equal-gaps fiction,
> and to say so out loud on the occasions you do use the mean anyway.

Let's load the file and read its types off the data itself.

> 💡 **Gemini Prompt:** "This cell loads a survey dataset and then prints, for
> every column, its stored data type and its number of distinct values: [paste
> the next cell]. Explain why the count of distinct values is only a HINT to
> whether a variable is nominal, ordinal, or numeric — and give me one case where
> that hint would mislead me."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the load prints 10,000 rows and 10 columns — the shape assert
>   guards it, so confirm the ✓ line appears.
> - [ ] Pick one column and decide its type from what it MEASURES, not from its
>   distinct-value count, then see whether Gemini's guess matches your reasoning.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the teaching resample and confirm its shape before trusting anything.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")
print()
print("Columns and stored dtypes:")
print(lapop.dtypes.to_string())
print()
print("Distinct values per column (a fast type hint):")
print(lapop.nunique().to_string())

### 🔍 Reading the Evidence

The `nunique` counts are a first clue to type, not a verdict. In the cell below,
answer three things. First: `municipality` has 107 distinct values and
`trust_police` has 7 — which is **nominal** and which is **ordinal**, and how do
the counts help you tell? Second: `self_efficacy_political` has only 4 distinct
values — does a small count of distinct values *prove* a variable is
categorical? (Careful: a numeric age rounded to decades would also show few
values.) Third: name one column where computing a **mean** would be a summary
that lies, and say why.

### YOUR ANSWER HERE:

**Nominal vs ordinal (municipality vs trust_police), and the tell:**

**Does few-distinct-values prove categorical?**

**A column where the mean would lie, and why:**

---

### 📝 Practice — repair three flawed descriptive claims

Each claim below is stated as if the data permit it. They do not. In the cell
that follows, **repair each** so it says exactly — no more — what a descriptive
summary of `lapop_brazil` could support. Watch for three failure modes: sneaking
past the data's edge (generalization), smuggling in cause, and forgetting the
resample caveat.

- **A.** "Brazilians trust the military more than the police."
- **B.** "Trust in the supreme court is low because the courts are corrupt."
- **C.** "The average municipality in Brazil scored 4.4 on trust in police."


### YOUR ANSWER HERE:

**A repaired:**

**B repaired:**

**C repaired:**

---

The type of a variable tells you which summary is *legal*. But even a legal
summary can hide the most important thing in the data. That is what the next
section is about.

---

### 🔮 Pause & Predict

We are about to plot the full distribution of `trust_military` — how the 10,000
answers spread across the 1–7 scale. Its average is a shade above 5. **Before
running the next cell**, sketch (in words) the shape you expect. Is it a tidy
hump centered near 5? Piled up at the high end? Split into camps? Commit to one
picture and one sentence of reasoning below.

### YOUR ANSWER HERE:

**The shape I expect (in words):**

**Why:**

---

> 💡 **Gemini Prompt:** "This cell draws the full distribution of a 1–7 survey
> item as a bar chart, marks the mean with a dashed line, and prints the mean,
> median, and mode: [paste the next cell]. Explain what it means when the mean
> lands on a scale point where very few respondents actually sit, and how the mode
> and the shape can tell a different story than the mean alone."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed counts against the chart — is the tallest bar really at
>   7, and does the dashed mean line (about 5.16) fall in a low valley?
> - [ ] Confirm the mean, median, and mode are three different numbers here before
>   accepting any claim that "the average describes the typical person."
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The reveal — run AFTER committing your prediction above.
counts = lapop["trust_military"].value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(counts.index, counts.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Trust in the military (1 = none, 7 = a lot)")
ax.set_ylabel("Number of respondents")
ax.set_title("Distribution of reported trust in the military\n(lapop_brazil teaching resample, n = 10,000)")
mean_tm = lapop["trust_military"].mean()
ax.axvline(mean_tm, color="#C44E52", linestyle="--", linewidth=2,
           label=f"mean = {mean_tm:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

print("Counts per scale point:")
print(counts.to_string())
print(f"\nmean = {mean_tm:.2f}   median = {lapop['trust_military'].median():.0f}   "
      f"mode = {lapop['trust_military'].mode().iloc[0]:.0f}")

In [ ]:
# Self-check: the mean sits in a valley, not on a peak — the point of this section.
assert abs(lapop["trust_military"].mean() - 5.16) < 0.01, "mean drifted — investigate"
assert lapop["trust_military"].mode().iloc[0] == 7, "mode should be 7"
assert counts.loc[5] < counts.loc[7], "expected far more 7s than 5s"
print("✓ Self-check passed: mean ≈ 5.16 lands between the scale points, "
      "mode is 7, and 7 is by far the most common answer.")

### 🔍 Reading the Evidence

The average was about 5.16 — but look where 5.16 actually falls. **Almost nobody
sits there.** The single most common answer is 7 (over 3,500 people), there is a
second pile at 6, and a stubborn lump of low-trust answers at 1. The mean, a
single number, splits the difference and lands in a valley between the camps. In
the cell below, write two things: what the *shape* tells a reader that the mean
of 5.16 hides, and one decision a researcher might get wrong by reporting only
the average.

### YOUR ANSWER HERE:

**What the shape shows that the mean hides:**

**A decision the average alone could get wrong:**

---

### Shape, at a glance: the boxplot

The bar chart showed you the full shape of trust in the military; a **boxplot** is
the compact version of that same shape — it marks the median and the middle half
of the answers, so a skew you saw in the bars turns into a lopsided box. Learn to
read the two together:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/shape_boxplot_map.png" width="78%"/></center>

*How a distribution's shape maps onto a boxplot: the same skew, read two ways.
(From Professor Moreira's QM 67000 Business Analytics slides.)*

---

## 3. Center, Spread, and One Honest Group Comparison

**Guiding question:** *when is an average a fair summary of a group — and when is
it a mask laid over two groups that have nothing in common?*

Every honest numeric summary reports **center and spread together** — the
typical value *and* how much answers vary around it. A mean with no spread is a
half-truth. And the most common analytical move — comparing two groups — is only
honest when it carries both, plus a claim boundary.

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/spread_vs_center.png" width="75%"/></center>

*Two groups with the same mean but very different spread — proof that a center
reported without its spread is only half the summary. (From Professor Moreira's
QM 67000 Business Analytics slides.)*

Below we split respondents by ideological self-placement (a 1–10 scale, where we
call 1–5 "left" and 6–10 "right") and compare average trust in the military.
This is a **descriptive group comparison**: real, useful, and sharply bounded.

> **A question that often comes up here:** *"If I already report the mean, why
> does the spread matter so much?"* Because two groups can share an identical mean
> and be nothing alike — one bunched tightly around it, the other pulled apart
> into extremes — and a decision that treats them as the same group would fail for
> one of them. The spread is what tells your reader whether the average describes
> a crowd standing together or the empty midpoint between two camps, which is
> exactly the trap the trust-in-military distribution sprang a moment ago.

### 🛠️ Run the Study: center, spread, and a two-group comparison

Run the cell. It reports the center and spread of `trust_military` overall, then
the same by ideological group. Read the numbers before you read the next
markdown cell.

> 💡 **Gemini Prompt:** "This cell reports the mean, median, standard deviation,
> and interquartile range of a trust item overall, then splits respondents into
> left and right ideological groups and compares their means: [paste the next
> cell]. Explain why a group mean reported WITHOUT its spread is only half a
> summary, and what the standard deviation adds to a two-group comparison."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the two group means against the printed difference — does right
>   minus left really equal the gap the cell reports?
> - [ ] Confirm each group's standard deviation prints alongside its mean; a
>   comparison of centers with no spreads is exactly the half-truth to avoid.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Center + spread overall, then a two-group descriptive comparison.
tm = lapop["trust_military"]
print("Trust in the military — overall")
print(f"  mean   = {tm.mean():.2f}")
print(f"  median = {tm.median():.0f}")
print(f"  std    = {tm.std():.2f}   (spread around the mean)")
print(f"  IQR    = {tm.quantile(.25):.0f} to {tm.quantile(.75):.0f}")
print()

lapop["ideo_group"] = np.where(lapop["ideology"] <= 5, "left (1–5)", "right (6–10)")
by_group = lapop.groupby("ideo_group")["trust_military"].agg(["mean", "std", "count"]).round(2)
print("Trust in the military — by ideological self-placement")
print(by_group.to_string())
gap = (by_group.loc["right (6–10)", "mean"] - by_group.loc["left (1–5)", "mean"])
print(f"\nDifference in group means (right − left): {gap:+.2f} points on the 1–7 scale")

In [ ]:
# Self-check: the comparison is real and its direction is stable.
assert abs(by_group.loc["left (1–5)", "mean"] - 4.78) < 0.01, "left mean drifted"
assert abs(by_group.loc["right (6–10)", "mean"] - 5.54) < 0.01, "right mean drifted"
assert by_group.loc["right (6–10)", "mean"] > by_group.loc["left (1–5)", "mean"]
print("✓ Self-check passed: right-leaning respondents report higher average "
      "military trust (≈5.54) than left-leaning respondents (≈4.78) in these data.")

### 🔍 Reading the Evidence — where this claim must stop

The gap is about 0.77 points, and it is real *in these data*. Now patrol its
edge. In the cell below, write the **most precise claim** this comparison
permits (it must name the resample and the scale), then write the **two
tempting sentences you must NOT write** — one that generalizes to all
Brazilians, and one that treats the ideology–trust gap as if ideology *caused*
the trust. Say, in one line each, exactly what evidence the forbidden sentences
would need that a description cannot supply.

### YOUR ANSWER HERE:

**Most precise permitted claim:**

**Forbidden generalization (and what it would need):**

**Forbidden causal claim (and what it would need):**

---

### ⚖️ Make a Design Choice — which center do you report?

`govt_pride` (a 1–7 pride-in-government scale) is heavily piled at the low end,
with a mode of 1 and a long tail upward. You must summarize it in one number for
a table. **Choose one:** report the **mean**, or report the **median**. In the
cell below, defend your choice in a short paragraph — say what your chosen number
tells the reader, what it hides, and why the *other* number would mislead more
for this particular shape.

> 💡 **Gemini Prompt:** "I have a 1–7 survey variable
> that is heavily skewed toward the low end (mode = 1, long tail upward). Explain,
> in plain language, when the mean and the median disagree for a skewed
> distribution, and which one better represents a 'typical' respondent — and when
> each can mislead."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Recompute both numbers yourself in the cell below (`.mean()` and
>   `.median()` on `govt_pride`) — confirm the AI's account matches what the data
>   actually do, not just what sounds right.
> - [ ] Check the direction: for a right-skewed variable, is the mean pulled
>   *above* or *below* the median? Verify against your own output.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE (recompute here, then defend your choice):

```python
# Recompute both — do not take the AI's word for it.
print(lapop["govt_pride"].mean(), lapop["govt_pride"].median())
```

**My choice (mean or median) and its defense:**

---

### 🎯 Project Transfer — your Table 1 skeleton

Every credible empirical paper opens with a "Table 1" that describes the sample
before any finding appears. Start yours now. In the cell below, sketch the
skeleton for *your* project's future data: list 4–6 variables you expect to
collect, and for each name its **type**, the **summary** you will report
(count / mean+sd / median+IQR), and — for your one key outcome — the **claim
boundary** your description will stop at. You are not filling in numbers yet; you
are committing to how you will describe honestly once numbers exist.

### YOUR ANSWER HERE:

| Variable | Type (nominal/ordinal/numeric) | Summary I will report | Claim boundary |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

**My key outcome and the exact sentence my description will stop at:**

---

### 🎟️ Claim Ticket

**Claim Ticket #13** — write your best descriptive claim from today's data,
worded to stop *exactly* at the data's edge: it must name the resample, report
center *and* spread (or a distribution shape), and contain no "plausibly" and no
"because."

### YOUR ANSWER HERE:

**My edge-of-the-data descriptive claim, one sentence:**

---

---

*(Meeting M14 picks up here.)*

## 4. Visualization That Tells the Truth — Same Data, Two Stories

> *"A chart is an argument. The reader's eye does the reasoning before their
> brain catches up — so if the picture overstates the numbers, you have made a
> claim you cannot defend, and you made it faster than anyone could check."*
> — a **journal reviewer** who rejects more figures than sentences

A distorted chart is not usually a fake chart. It is an *accurate* chart drawn to
make the eye conclude more than the numbers support. The most common trick is the
**truncated axis**: start the y-axis partway up, and a small real difference
balloons into a dramatic one. Add a **cherry-picked framing** — a loaded title,
a convenient comparison — and the picture argues a case the data never made.

We will build a deliberately misleading chart from a real `lapop_brazil`
difference, then repair it step by step. Same numbers, both times. Only the
honesty changes.

In [ ]:
# Build the two group means we will chart (govt_pride by ideological group).
by_pride = lapop.groupby("ideo_group")["govt_pride"].mean()
left_val = by_pride.loc["left (1–5)"]
right_val = by_pride.loc["right (6–10)"]
print(f"Average government pride (1–7 scale):")
print(f"  left (1–5):  {left_val:.2f}")
print(f"  right (6–10): {right_val:.2f}")
print(f"  real difference: {right_val - left_val:+.2f} points on a 7-point scale")

assert abs(left_val - 2.68) < 0.01 and abs(right_val - 3.29) < 0.01, "means drifted"
print("✓ Both group means below the scale midpoint of 4 — government pride is low in both groups.")

### 🔮 Pause & Predict

The real difference above is about 0.62 points on a 1–7 scale — modest, and both
groups sit *below* the scale's midpoint (4). **Before running the next cell**,
predict: if someone wanted to make the right-hand group look dramatically prouder,
where would they start the y-axis, and roughly how many times taller could they
make one bar look than the other? Commit a number below.

### YOUR ANSWER HERE:

**Where they'd start the axis, and the exaggeration factor I predict:**

---

> 💡 **Gemini Prompt:** "This cell charts two group means that differ by about
> 0.62 on a 1–7 scale, but it starts the y-axis at 2.5 and gives the chart a
> punchy title: [paste the next cell]. Explain, step by step, how starting the
> axis above zero turns a small real gap into a bar that looks several times
> taller, and how the printed exaggeration factor is computed."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed exaggeration factor against the bars you see — does the
>   right bar really look several times the left, for a real gap of only 0.62?
> - [ ] Confirm both group means sit below the scale midpoint of 4, so the
>   "surges" title is claiming more than the numbers support.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The DISTORTED chart — this is how a 0.62-point gap gets sold as a chasm.
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["Left", "Right"], [left_val, right_val],
       color=["#8899AA", "#C44E52"], edgecolor="white")
ax.set_ylim(2.5, 3.4)                       # truncated axis: the whole trick
ax.set_ylabel("Government pride")           # no scale range shown
ax.set_title("Government pride SURGES on the political right")  # cherry-picked framing
plt.tight_layout()
plt.show()

# How big is the visual lie? Compare bar heights ABOVE the truncated floor.
floor = 2.5
exaggeration = (right_val - floor) / (left_val - floor)
print(f"Truncated at {floor}: the right bar is drawn {exaggeration:.1f}× taller "
      f"than the left — for a real gap of only {right_val - left_val:.2f} points.")

### 🛠️ Hands-On: repair the chart, one distortion at a time

Now un-distort it. The cell below redraws the *same two numbers* honestly, fixing
three things at once: (1) the axis runs the full 1–7 scale the variable actually
lives on; (2) a reference line marks the scale midpoint so "high" and "low" are
visible; (3) the title describes instead of editorializing, and the caption
carries the resample caveat. Run it, then compare the two pictures.

> 💡 **Gemini Prompt:** "This cell redraws the SAME two numbers honestly — a full
> 1–7 axis, a reference line at the scale midpoint, a descriptive title, and a
> caption carrying the resample caveat: [paste the next cell]. For each of those
> four changes, tell me which specific piece of the earlier misleading impression
> it removes."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Put the two charts side by side: on the honest one, does the 0.62 gap now
>   look small, and do both bars sit below the midpoint line?
> - [ ] Confirm the numbers are identical to the distorted chart — only the
>   drawing changed, which is the whole point of the exercise.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The HONEST chart — same data, truthful drawing.
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["Left (1–5)", "Right (6–10)"], [left_val, right_val],
       color=["#8899AA", "#4C72B0"], edgecolor="white")
ax.set_ylim(1, 7)                                  # full scale, no truncation
ax.axhline(4, color="gray", linestyle=":", linewidth=1.5, label="scale midpoint (4)")
ax.set_ylabel("Average government pride (1 = none, 7 = a lot)")
ax.set_title("Average government pride by ideological self-placement")
ax.legend(loc="upper right")
ax.text(-0.4, 0.4, "lapop_brazil teaching resample, n = 10,000 — for planning, not population claims",
        fontsize=8, color="gray")
plt.tight_layout()
plt.show()

print(f"Same numbers ({left_val:.2f} vs {right_val:.2f}) — now the reader sees "
      f"that both groups report LOW pride, and the gap is small.")

### 🔍 Reading the Evidence — the impression is also a claim

Look at the two figures side by side. The numbers never changed. In the cell
below, write three things: what *visual* claim the first chart makes that the
numbers do not support; which single repair (axis, framing, or context) did the
most to defuse it; and — the uncomfortable one — whether a technically-accurate
but misleading chart counts as a lie, and who you think is responsible when a
reader is fooled, the maker or the reader.

### YOUR ANSWER HERE:

**The unsupported visual claim in the distorted chart:**

**The repair that mattered most:**

**Is an accurate-but-misleading chart a lie? Who is responsible?**

---

### The honest-visualization checklist

Keep this next to every figure you make for the rest of the course:

- **Axis:** does it start where the scale starts (or at a defensible zero), so
  bar heights are proportional to the numbers?
- **Framing:** does the title *describe* the data rather than argue a verdict?
- **Context:** are the sample size, the scale range, and any resampling/missing
  cases visible on or beside the figure?
- **Bins & comparison:** for histograms, are the bins honest (not chosen to
  manufacture a hump)? Is the comparison the fair one, not the flattering one?
- **Re-drawability:** could a careful reader reconstruct your chart from your
  numbers and get the same picture?

For a public gallery of axis and dataviz distortions to study, the *Calling
Bullshit* project keeps an open case index and tools page:
[callingbullshit.org/tools](https://callingbullshit.org/tools/) and
[callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html).

> **A question that often comes up here:** *"Is it always wrong to start a bar
> chart's axis above zero?"* For **bar** charts, almost always — a bar encodes its
> value by length, so a truncated baseline makes the length itself lie. The honest
> exception is a **line** chart tracking change over time, where the reader judges
> the slope rather than compares lengths, and a zoomed axis can reveal a real
> trend without distorting it. The test is never the axis alone; it is whether the
> visual quantity the eye compares still matches the numbers.

### One more distortion to know: the pie chart

A truncated axis exaggerates a gap; a pie chart tends to hide one. People compare
*lengths* far more accurately than they compare *angles* or *areas*, so slices
that are easy to draw are hard to read — and a 3-D tilt makes it worse by throwing
the front slice toward the viewer. The same numbers drawn as a bar chart hand the
eye the comparison a pie only asks it to guess:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/pie-vs-bar.png" width="80%"/></center>

*The same numbers drawn as a pie and as a bar chart — the bar lengths are easy to
compare, the pie slices are not. (From Professor Moreira's QM 67000 Business
Analytics slides.)*

The most famous case is a 2008 Apple product keynote:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/steve-jobs-pie.jpg" width="70%"/></center>

*A 2008 Apple keynote 3-D pie chart: the 19.5% slice tilted toward the front is
drawn to look nearly as large as the 39% slice behind it — perspective distorting
the comparison. (Apple keynote photograph, 2008 — a classic visualization-criticism
case, used in Professor Moreira's slides.)*

---

### 📝 Practice — name the distortion

Three figures are described below. For each, name the distortion mechanism (from
the checklist) and the one repair that would make it honest. Write your answers
in the cell that follows.

- **A.** A bar chart of two campus dining halls' satisfaction (4.2 vs 4.4 out of
  5) with a y-axis running from 4.0 to 4.5, titled "Hillenbrand crushes the
  competition."
- **B.** A histogram of exam scores whose bins are 0–59, 60–61, 62–63, 64–100 —
  making the middle look like a towering spike.
- **C.** A line chart of one club's membership over the three months it happened
  to grow, cropped from a two-year record in which it mostly fell.


### YOUR ANSWER HERE:

**A — distortion + repair:**

**B — distortion + repair:**

**C — distortion + repair:**

---

### 🎯 Project Transfer — the checklist becomes a standing rubric row

The honest-viz checklist is not just for today; it is a permanent row on your
poster rubric (M10–M12). In the cell below, name the **one figure** you already
expect your project to need, and pre-commit: which checklist item is it *most*
at risk of failing, and what will you do to keep it honest? Writing this now is
cheaper than discovering it at the poster session.

### YOUR ANSWER HERE:

**The figure my project will need:**

**The checklist item it's most at risk of failing, and my safeguard:**

---

### 🎟️ Claim Ticket

**Claim Ticket #14** — name one distortion you will actively avoid in your own
project's figures, and the concrete drawing decision that will prevent it.

### YOUR ANSWER HERE:

**Distortion I'll avoid + the drawing decision that prevents it:**

---

## 5. Wrap-Up

Across two meetings you did the unglamorous, load-bearing work of the first
approach: you matched summaries to variable types, read distributions instead of
trusting averages, reported center *with* spread, ran one honest group comparison,
and learned to see a chart as the argument it is — repairing a truncated-axis lie
back into a truthful picture of the very same numbers. Underneath all of it ran
one discipline: description stops at the data's edge, and the resample caveat
rides along with every number you quote.

> **"A description is honest when a careful reader, given only your summary,
> would picture the data the way it actually is — no larger, no cleaner, and no
> more certain than it earned."**

On Monday, nb07 opens MIDA's Data strategy and asks the question description
cannot: *whose* data is this? You will draw real samples from a voter file, watch
a convenience sample lie, and start the M05 Data Strategy that decides who gets
into your evidence in the first place. Bring your Table 1 skeleton — it is about
to meet the sampling frame.

---

## 6. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 15 'Observational: descriptive' | observational descriptive designs | variable types, distributions, group summaries, the descriptive claim boundary | adapted (concept-level, honors non-quant audience)*
- *lapop_brazil.csv | rdss package data | trust/pride/ideology items summarized and charted | adapted (10,000-row resample-with-replacement; caveat taught explicitly as a provenance lesson)*
- *fresh | honest-visualization checklist + truncated-axis repair | — | newly-constructed-from-source-concept*
- *callingbullshit.org (public index) | misleading-axes / dataviz material | linked as optional study, index pages only | referenced*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, ch. 15 'Observational: descriptive'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit* — misleading axes /
  dataviz case index: [callingbullshit.org/tools](https://callingbullshit.org/tools/).

---

<center>

Thank you!

</center>